# Customer Conversion Analytics — Funnel by Segment & Retention

Breaks the conversion funnel down by device and acquisition channel instead
of reporting one blended number, and looks at which customer segments
actually come back. See `docs/methodology.md`.

In [1]:
import pandas as pd

segments = pd.read_csv("../data/website_sessions_by_segment.csv")
customers = pd.read_csv("../data/customers.csv", parse_dates=["first_purchase"])
print("segments:", segments.shape, "| customers:", customers.shape)


segments: (216, 7) | customers: (320, 6)


## 1. Funnel by device — where the blended average hides the real story

In [2]:
by_device = segments.groupby("device").agg(
    sessions=("sessions","sum"), add_to_cart=("add_to_cart","sum"),
    checkout=("checkout","sum"), purchases=("purchases","sum"),
)
by_device["conversion_rate_pct"] = (100*by_device["purchases"]/by_device["sessions"]).round(2)
by_device["checkout_completion_pct"] = (100*by_device["checkout"]/by_device["add_to_cart"]).round(2)
by_device.sort_values("conversion_rate_pct", ascending=False)


Desktop converts more than twice as well as mobile, and the gap is mostly in checkout completion, not add-to-cart. That's a mobile checkout problem, not a mobile traffic-quality problem.

## 2. Funnel by acquisition channel

In [3]:
by_channel = segments.groupby("acquisition_channel").agg(sessions=("sessions","sum"), purchases=("purchases","sum"))
by_channel["conversion_rate_pct"] = (100*by_channel["purchases"]/by_channel["sessions"]).round(2)
by_channel.sort_values("conversion_rate_pct", ascending=False)


## 3. Repeat purchase rate by segment

In [4]:
repeat = customers.groupby(["country","customer_type"]).agg(
    customers=("customer_id","count"), repeat_customers=("repeat_purchase","sum")
)
repeat["repeat_rate_pct"] = (100*repeat["repeat_customers"]/repeat["customers"]).round(1)
repeat.sort_values("repeat_rate_pct", ascending=False).head(8)


## 4. Acquisition-month cohorts (with a recency caveat)

In [5]:
c = customers.copy()
c["cohort_month"] = c["first_purchase"].dt.to_period("M").astype(str)
cohorts = c.groupby("cohort_month").agg(customers=("customer_id","count"), repeat_customers=("repeat_purchase","sum"))
cohorts["repeat_rate_pct"] = (100*cohorts["repeat_customers"]/cohorts["customers"]).round(1)
cohorts


Recent cohorts (2026-05, 2026-06) show a lower repeat rate — but they've
also had less time to make a second purchase than the 2025-08 cohort. This
is right-censoring, not necessarily a real drop in loyalty. A proper
cohort analysis would measure repeat rate within a fixed window (e.g. 90
days) per cohort instead of "as of today", which is the honest caveat to
put on this table before anyone acts on it.

## 5. Key findings

1. **Desktop converts at 6.60% versus 3.13% on mobile**, and the gap is
   concentrated in checkout completion (69.1% vs. 51.7%), not in getting
   people to add a product to cart. Fix the checkout flow, not the whole
   mobile site.
2. **Referral converts best per session (4.82%)** across acquisition
   channels, ahead of organic search (4.72%) and paid search (4.56%),
   despite the smallest volume.
3. **The worst-converting segment is mobile + paid search (3.04%)** —
   paying to send mobile traffic into the funnel's weakest combination is
   the least efficient use of acquisition budget in this dataset.
4. **Repeat purchase rate by country/segment varies from ~12% to ~52%** —
   worth investigating what BE B2C customers (highest repeat rate) are
   experiencing differently before assuming it's a fluke.